# v22 ヒューリスティックモデル 単体 Kaggle 提出ノート(自己完結・外部import無し / PFバンク平均)

**内容**: test の各評価行(TVT_input=NaN)に対し、
**選んだ複数のPFパラメータ(PF_BANK)で各々推論 → 平均** → カテゴリ別 scale + beam + hold + apply_pp → submission.csv。

- 他モデル(GBM等)は使わない。**ヒューリスティックモデル単体**の LB 評価用。
- **全コードをこのノート内に記述**(別ファイル import 無し)。PF/beam エンジンはインライン。
- **PF_BANK** で PFパラメータを複数選ぶと、その全部で推論して平均する(1個なら単体)。候補=v19バンク r0〜r4。
- カテゴリ = `code = int(n_eval>4840) + 2*searchsorted((136.73,185.51), z_span)`。scale はカテゴリ別(BEST_SCALE)。
- **beam / hold**(7.191 selector 由来, codes 2,4,5 / 全code)+ **apply_pp**(tau=85 ランプ + sg_smooth)。
- 教訓: PFを坑井別にフルチューニング(cat分岐)は過学習(LB悪化)。**単一〜少数の頑健PFを平均 + 軽ヒューリスティック**が正解。
- test は隠しセットに置換され rerun。**test 実在列(MD,X,Y,Z,GR,TVT_input + typewell)だけ**から生成。


In [ ]:
# ===== import =====
import os, glob, json, warnings
from collections import defaultdict
import numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.signal import savgol_filter
from numba import njit
warnings.filterwarnings("ignore"); np.seterr(all="ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32
print("device:", DEVICE, "| torch:", torch.__version__)


In [ ]:
# ===== データパス自動判定(sample_submission.csv を含むルートを探す) =====
CANDS = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    "C:/Users/kosaka256/Documents/rogii_claude/rogii-wellbore-geology-prediction",
]
COMP = next((p for p in CANDS if os.path.exists(os.path.join(p, "sample_submission.csv"))), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else CANDS[-1]
COMP = Path(COMP); TEST = COMP / "test"
print("COMP:", COMP, "| test exists:", TEST.exists())


In [ ]:
# ===== 設定 =====
N_EVAL_THRESHOLD  = 4840.0
Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
USE_PHYS = True
SEEDS    = 128         # アンサンブル seed 数(r0単体 7.701 は 128。32/8 も試す価値)
CHUNK    = 100         # GPUバッチ坑井数(OOMなら下げる)

# ===== PFバンク: 選んだ全PFで推論して平均 =====
PF_LIB = {
    'r0': {'n_particles': 500, 'likelihood_scale': 6.265873932344672, 'init_pos_std': 1.0481897997156377, 'init_rate_std': 0.004385968881669577, 'mom': 0.9995659517817024, 'vn': 0.0009771590366402883, 'pn': 0.02193433099852571, 'rp': 1.266960649917433, 'rr': 0.0003377853906820054, 'resamp': 0.05083194815648496, 'gr_sig_min': 10.293696795145928, 'gr_sig_max': 49.46806691781805, 'gr_sig_def': 129.97956203556075, 'gr_sig_mult': 3.412937310824599, 'tvt_clip_margin': 159.57810929194702, 'rate_clip': 0.0, 'gr_power': 1.5371545258734838, 'hgr_smooth_r': 1, 'tw_gr_smooth_r': 0, 'jump_prob': 0.002, 'jump_std': 5.0, 'phys_sig': 4.0313262188520795, 'use_phys': True},
    'r1': {'n_particles': 800, 'likelihood_scale': 20.937794817665264, 'init_pos_std': 1.61942367399371, 'init_rate_std': 0.0011008246369334533, 'mom': 0.99957579312239, 'vn': 0.0010918186495654705, 'pn': 0.011816846470879912, 'rp': 0.7987525607018457, 'rr': 0.0003521085638480051, 'resamp': 0.05187694934697306, 'gr_sig_min': 8.29719914421004, 'gr_sig_max': 54.79764654360716, 'gr_sig_def': 122.46292926802585, 'gr_sig_mult': 3.6189633272014663, 'tvt_clip_margin': 50.378942201651405, 'rate_clip': 1.5, 'gr_power': 2.3368520856274184, 'hgr_smooth_r': 3, 'tw_gr_smooth_r': 0, 'jump_prob': 0.0, 'jump_std': 20.0, 'phys_sig': 2.709572647501733, 'use_phys': True},
    'r2': {'n_particles': 400, 'likelihood_scale': 6.63806132226747, 'init_pos_std': 1.99444683836238, 'init_rate_std': 0.0005392083746546821, 'mom': 0.9999782165028804, 'vn': 0.0014652810497252174, 'pn': 0.0049467341418455504, 'rp': 0.520101375920893, 'rr': 0.0002445451847403657, 'resamp': 0.05007811818705463, 'gr_sig_min': 11.014474785658665, 'gr_sig_max': 52.42081390277763, 'gr_sig_def': 143.45638066094898, 'gr_sig_mult': 3.414209648945256, 'tvt_clip_margin': 157.5013420843151, 'rate_clip': 0.0, 'gr_power': 1.6501931262766802, 'hgr_smooth_r': 3, 'tw_gr_smooth_r': 1, 'jump_prob': 0.002, 'jump_std': 5.0, 'phys_sig': 1.9464074823760222, 'use_phys': True},
    'r3': {'n_particles': 800, 'likelihood_scale': 57.1951554638748, 'init_pos_std': 0.5425696395634219, 'init_rate_std': 0.0019178997994233119, 'mom': 0.9995113027332957, 'vn': 0.001242820074957157, 'pn': 0.035201224487458604, 'rp': 0.11781303647499938, 'rr': 0.00010708342158430313, 'resamp': 0.07782686296612414, 'gr_sig_min': 6.198297071230761, 'gr_sig_max': 48.7846557244348, 'gr_sig_def': 91.50844322602568, 'gr_sig_mult': 3.7649103884187873, 'tvt_clip_margin': 144.52478835523414, 'rate_clip': 3.0, 'gr_power': 2.3022100967797976, 'hgr_smooth_r': 0, 'tw_gr_smooth_r': 1, 'jump_prob': 0.0, 'jump_std': 5.0, 'phys_sig': 2.5902973341149775, 'use_phys': True},
    'r4': {'n_particles': 500, 'likelihood_scale': 16.330356584019487, 'init_pos_std': 0.17582775028903652, 'init_rate_std': 0.008379439780636207, 'mom': 0.999969910450737, 'vn': 0.0009342864713995873, 'pn': 0.007647461427640706, 'rp': 0.1388057043012901, 'rr': 0.0006012621586488701, 'resamp': 0.07060130224944136, 'gr_sig_min': 10.888923159937743, 'gr_sig_max': 49.82454650951251, 'gr_sig_def': 82.84180461559971, 'gr_sig_mult': 2.711994048067299, 'tvt_clip_margin': 170.91986146579927, 'rate_clip': 0.0, 'gr_power': 2.8115024149784986, 'hgr_smooth_r': 2, 'tw_gr_smooth_r': 1, 'jump_prob': 0.002, 'jump_std': 5.0, 'phys_sig': 4.2423220341326635, 'use_phys': True},
}

PF_BANK = ["r0"]       # ← ここで選ぶ。複数指定(例 ["r0","r4"])すると全部で推論して平均。

# カテゴリ別ベスト scale(温度)
BEST_SCALE = {0: 8.0, 1: 8.0, 2: 5.0, 3: 20.0, 4: 12.0, 5: 5.0}

# 後処理 apply_pp(v9/v20 と同形。GBM無しなので PF残差のみ→w_pf は無効、tau ランプ + alpha + sg_smooth)
APPLY_PP  = True
PP_PARAMS = {"alpha": 1.0, "tau": 85, "w_pf": 0.09}
SG_W, SG_P = 17, 3

# 7.191 selector の beam / hold(per-code 重み)。pred=(1-bw)*PF+bw*beam →(1-hw)*pred+hw*last_known_tvt
USE_BEAM = True
USE_HOLD = True
BEAM_W = {0: 0.0, 1: 0.0, 2: 0.2, 3: 0.0, 4: 0.05, 5: 0.2}
HOLD_W = {0: 0.2, 1: 0.15, 2: 0.15, 3: 0.15, 4: 0.05, 5: 0.05}
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1), (10, 14.0, 90.0, 5),
    (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3), (15, 25.0, 180.0, 2), (20, 30.0, 200.0, 2),
    (15, 10.0, 80.0, 4), (25, 6.0, 50.0, 3), (10, 40.0, 300.0, 1), (12, 18.0, 120.0, 5),
    (30, 8.0, 70.0, 2), (10, 50.0, 400.0, 0),
]
assert all(k in PF_LIB for k in PF_BANK), "PF_BANK に PF_LIB に無いキーがある"
print("PF_BANK:", PF_BANK, "| seeds:", SEEDS, "| beam:", USE_BEAM, "hold:", USE_HOLD, "pp:", APPLY_PP)


In [ ]:
# ===== 物理PFエンジン(v22 と同一・インライン) + test 用入力ビルダー =====
def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def well_code(n_eval, z_span):
    n_bin = int(n_eval > N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(Z_SPAN_THRESHOLDS, z_span, side="right"))
    return n_bin + 2*z_bin

def build_test(wid):
    """test 坑井の PF 入力素材 + 評価行 id を返す(TVT 真値は使わない)。無効なら None。"""
    hw = pd.read_csv(TEST / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TEST / f"{wid}__typewell.csv").sort_values("TVT")
    if "TVT_input" not in hw.columns or "TVT" not in tw.columns:
        return None
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0 or len(kn) < 1:
        return None
    tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    if len(tw_tvt) < 3:
        return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_filled = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    ev_pos = hw.index.get_indexer(ev.index); kpos = hw.index.get_indexer(kn.index)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    z_ev = ev["Z"].to_numpy(float)
    n_eval = int(len(z_ev)); z_span = float(np.nanmax(z_ev)-np.nanmin(z_ev)) if len(z_ev) else 0.0
    ev_ids = [f"{wid}_{i}" for i in ev.index]
    last_md = float(kn["MD"].iloc[-1]); last_tvt = float(kn["TVT_input"].iloc[-1])
    return dict(md=ev["MD"].to_numpy(float), z=z_ev, gr_filled=gr_filled, ev_pos=ev_pos, kpos=kpos,
                kn_tvtin=kn["TVT_input"].to_numpy(float), tw_tvt=tw_tvt, tw_gr=tw_gr, tw_fb=tw_fb,
                ls=ls, ir=ir, n_eval=n_eval, z_span=z_span, ev_ids=ev_ids,
                last_known_md=last_md, last_known_tvt=last_tvt)

def make_inputs(c, P):
    tw_gr_pf = _smooth(c["tw_gr"], c["tw_fb"], P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(c["tw_tvt"], tw_gr_pf)
    gr_sm = _smooth(c["gr_filled"], c["tw_fb"], P["hgr_smooth_r"])
    if len(c["kpos"]) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[c["kpos"]] - np.interp(c["kn_tvtin"], c["tw_tvt"], tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs*P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"]+1e-6)))
    return dict(md=c["md"], z=c["z"], gr=gr_sm[c["ev_pos"]].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=c["ls"], ir=c["ir"], N=int(P["n_particles"]))


In [ ]:
# ===== 物理PF コア(GPU) + パディング(v22 と同一) =====
def _pf_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = bool(P.get("use_phys", USE_PHYS))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS*rn(); rate = ir[:, None] + IR*rn()
    w = torch.full((B, N), 1.0/N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype)-1)*step + CLIP
    glast = torch.gather(grid, 1, (glen-1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur-pm).clamp_min(1.0)
        rate_n = ALPHA*rate + RN*rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n*dm[:, None] + PN*rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD*rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n-zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt+zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos-zi; ii = (v-vmin[:, None])/step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen-1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen-2).clamp_min(0)[:, None])
        t = ii-i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c+1)
        eg = gA*(1-t)+gB*t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None]-eg)/gs[:, None]).abs(); dp = d**GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5*dp), zero).clamp_min(1e-300)
        avg = (w*lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate-dipp)/PHYS_SIG; lkp = torch.exp(-0.5*dphys*dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w*lk*lkp, w*lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new/ws.clamp_min(1e-300), torch.full_like(w, 1.0/N))
        ne = (w*w).sum(1); need = act & ((1.0/ne) < (RESAMP*N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype)*(1.0/N)
            u = u0 + arN[None, :]/N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N-1)
            pos_rs = torch.gather(pos, 1, idx) + RP*rn(); rate_rs = torch.gather(rate, 1, idx) + RR*rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0/N), w)
        pts[:, i] = (w*(pos-zi)).sum(1); pm = torch.where(act, cur, pm)
    return pts, log_lik

def _pad(inps, device, dtype=DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn-1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G-1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_pf_scale(inps, P, seed, n_seeds, chunk, scale):
    """PFを1回走らせ(S seeds)、scale(温度)で seed 尤度加重 → 各坑井の評価行TVT予測を返す。"""
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); out = [None]*W
    for c0 in range(0, W, chunk):
        sub = inps[c0:c0+chunk]; w = len(sub); pad = _pad(sub, DEVICE)
        rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=DEVICE); gen.manual_seed(int(seed))
        pts, ll = _pf_core(rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
                           pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
                           pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
                           P, N, DEVICE, gen)
        pts = pts.view(w, S, -1).double().cpu().numpy(); ll = ll.view(w, S).cpu().numpy()
        for j in range(w):
            T = len(sub[j]["md"]); pr = pts[j, :, :T]; llw = ll[j]
            if np.isfinite(llw).any():
                mx = np.nanmax(llw[np.isfinite(llw)])
                ww = np.exp(np.where(np.isfinite(llw), llw-mx, -np.inf)/max(scale, 1e-6))
                ww = ww/ww.sum() if ww.sum() > 1e-300 else np.full(S, 1.0/S)
            else:
                ww = np.full(S, 1.0/S)
            out[c0+j] = (ww[:, None]*pr).sum(0)
    if DEVICE == "cuda": torch.cuda.synchronize()
    return out

# ===== beam search(7.191 移植, numba)=====
@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS*6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv-tw_gr[ni])**2/es + mc*(d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i+1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n-1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

def _nn(arr, val): return int(np.abs(arr - val).argmin())

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, int(bs), float(mc), float(es))
    return tw_tvt[path].astype(np.float32)

def run_beam(c):
    tw_tvt = c["tw_tvt"].astype(np.float64)
    fb = float(np.nanmean(c["tw_gr"])) if np.isfinite(np.nanmean(c["tw_gr"])) else 0.0
    tw_gr = np.where(np.isfinite(c["tw_gr"]), c["tw_gr"], fb).astype(np.float64)
    hgr = c["gr_filled"][c["ev_pos"]]; last_tvt = c["last_known_tvt"]
    res = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r) in BEAM_CONFIGS]
    return np.stack(res, 0).mean(0).astype(np.float64)

def post_process(pred, c):
    """apply_pp(PF残差のみ: md=pd_=PF残差 → w_pf無効) + sg_smooth。pred:坑井の評価行TVT配列。"""
    pred = np.asarray(pred, dtype=np.float64)
    if not APPLY_PP:
        return pred
    md_since = np.maximum(c["md"] - c["last_known_md"], 0.0)
    d = pred - c["last_known_tvt"]
    if PP_PARAMS.get("tau"):
        d = d * (1.0 - np.exp(-md_since / float(PP_PARAMS["tau"])))
    d = d * float(PP_PARAMS.get("alpha", 1.0))
    out = c["last_known_tvt"] + d
    n = len(out); wl = min(SG_W, n)
    if wl % 2 == 0: wl -= 1
    if wl >= SG_P + 2:
        out = savgol_filter(out, wl, SG_P)
    return out


In [ ]:
# ===== test 推論: カテゴリ分け → PF_BANK 全PFで推論して平均 → beam/hold/apply_pp =====
import time
wids = sorted(set(Path(p).stem.replace("__horizontal_well", "")
                  for p in glob.glob(str(TEST / "*__horizontal_well.csv"))))
print("test wells:", len(wids))

CACHE = {}; CODE = {}
for wid in wids:
    try:
        c = build_test(wid)
    except Exception:
        c = None
    if c is not None:
        CACHE[wid] = c; CODE[wid] = well_code(c["n_eval"], c["z_span"])
print("usable wells:", len(CACHE))

groups = defaultdict(list)
for wid in CACHE: groups[CODE[wid]].append(wid)
print("カテゴリ別 well 数:", {k: len(groups[k]) for k in sorted(groups)})

rows = []; t0 = time.time()
for code in sorted(groups):
    wl = groups[code]; scale = BEST_SCALE[code]
    bw = BEAM_W.get(code, 0.0) if USE_BEAM else 0.0
    hw = HOLD_W.get(code, 0.0) if USE_HOLD else 0.0
    # --- PFバンク平均: 選んだ各PFで推論して平均 ---
    acc = None
    for bi, key in enumerate(PF_BANK):
        P = dict(PF_LIB[key]); P.setdefault("use_phys", USE_PHYS)
        seed = 100*code + 1000*bi                        # PFごとにseedをずらして多様性
        inps = [make_inputs(CACHE[w], P) for w in wl]
        preds = run_pf_scale(inps, P, seed, SEEDS, CHUNK, scale)
        if acc is None:
            acc = [np.asarray(p, dtype=np.float64) for p in preds]
        else:
            for i, p in enumerate(preds): acc[i] = acc[i] + np.asarray(p, dtype=np.float64)
    acc = [a / len(PF_BANK) for a in acc]
    # --- beam(PF非依存, codes 2,4,5)→ hold → apply_pp/sg ---
    for w, base in zip(wl, acc):
        c = CACHE[w]; pred = base.copy()
        if bw > 0:
            beam = run_beam(c); m = min(len(pred), len(beam)); pred[:m] = (1.0-bw)*pred[:m] + bw*beam[:m]
        if hw > 0:
            pred = (1.0-hw)*pred + hw*c["last_known_tvt"]
        pred = post_process(pred, c)
        ids = c["ev_ids"]; n = min(len(ids), len(pred))
        rows.extend((ids[k], float(pred[k])) for k in range(n))
    print(f"cat{code}: wells={len(wl)} scale={scale} beam={bw} hold={hw} pf_bank={len(PF_BANK)} ({time.time()-t0:.0f}s)")
pred_df = pd.DataFrame(rows, columns=["id", "tvt"])
print("predicted rows:", len(pred_df))


In [ ]:
# ===== 提出ファイル作成 =====
ss = pd.read_csv(COMP / "sample_submission.csv")
sub = ss[["id"]].merge(pred_df, on="id", how="left")
fill = float(np.nanmedian(pred_df["tvt"])) if len(pred_df) else 0.0
n_missing = int(sub["tvt"].isna().sum())
sub["tvt"] = sub["tvt"].fillna(fill)
sub.to_csv("submission.csv", index=False)
print("submission:", sub.shape, "| filled(missing):", n_missing, "| fill_value:", round(fill, 3))
sub.head()
